# Dataset Exploration for Master Thesis Project Definition

This notebook helps you analyze a local TensorFlow dataset and decide which kind of thesis project is most suitable based on the data structure and labels.

It is designed for a dataset loaded as:

```python
import tensorflow_datasets as tfds
builder = tfds.builder_from_directory("path/to/robo_ai_u_r5e/1.0.0")
ds = builder.as_dataset(split="train")
```

The goal is not to guess the project immediately, but to inspect the data and connect it to realistic thesis directions such as classification, object detection, action prediction, anomaly detection, and robotic perception.

In [ ]:
import os
import glob
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds


def find_dataset_path():
    candidates = [
        os.environ.get("ROBO_AI_DATASET_PATH"),
        r"C:\Users\Nahid\Test\robo_ai_u_r5e\1.0.0",
        r"C:\Users\Nahid\robo_ai_u_r5e\1.0.0",
        r"C:\Users\Nahid\Test\path\to\robo_ai_u_r5e\1.0.0",
        r"C:\Users\Nahid\path\to\robo_ai_u_r5e\1.0.0",
        r".\robo_ai_u_r5e\1.0.0",
        r".\data\robo_ai_u_r5e\1.0.0",
    ]
    seen = set()
    for c in candidates:
        if not c or c in seen:
            continue
        seen.add(c)
        if os.path.exists(c):
            return c
    matches = glob.glob(os.path.join("**", "robo_ai_u_r5e", "1.0.0"), recursive=True)
    for m in matches:
        if os.path.isdir(m):
            return m
    return None

DATASET_PATH = find_dataset_path()
if DATASET_PATH is None:
    raise FileNotFoundError(
        "Dataset not found in this environment. Please set the actual path in DATASET_PATH or provide the dataset folder. Expected folder: robo_ai_u_r5e/1.0.0"
    )

print("Using dataset path:", DATASET_PATH)
builder = tfds.builder_from_directory(DATASET_PATH)
print("Dataset builder loaded:", builder.name)
print("Dataset info:")
print(builder.info)

ds = builder.as_dataset(split="train")
print("Dataset type:", ds)
print("Number of examples in train split:", ds.cardinality().numpy())

## Step 2: Preview actual examples

Now you can look at a few examples to confirm whether the dataset contains images, states, actions, text, or multimodal data.

In [ ]:
NUM_EXAMPLES = 3
for i, example in enumerate(ds.take(NUM_EXAMPLES)):
    print(f"\n--- Example {i} ---")
    if isinstance(example, dict):
        for key, value in example.items():
            if hasattr(value, "shape"):
                print(f"{key}: shape={value.shape}, dtype={value.dtype}")
                if tf.is_tensor(value):
                    try:
                        print("  first values:", value.numpy()[:5] if value.shape.rank == 1 else value.numpy()[:2])
                    except Exception:
                        pass
            else:
                print(f"{key}: {value}")
    else:
        print(example)

## Step 3: Check whether the dataset is supervised or sequential

Look for keys such as `image`, `observation`, `state`, `label`, `action`, `target`, `instruction`, `depth`, `mask`, `reward`, and `trajectory`. This is one of the most important parts for deciding the thesis direction.

In [ ]:
example = next(iter(ds))
if isinstance(example, dict):
    keys = list(example.keys())
    print("All keys in sample:", keys)

    key_map = {k.lower(): k for k in keys}
    possible_targets = [
        "label", "labels", "class", "classes", "target", "targets",
        "action", "actions", "instruction", "goal", "trajectory",
        "image", "images", "observation", "state", "states"
    ]
    found = []
    for item in possible_targets:
        if item in key_map:
            found.append(item)
    print("Likely relevant fields:", found)
else:
    print("The dataset element is not a dictionary.")

## Step 4: Analyze labels or target distributions

If the dataset contains a class label, this is a strong sign for classification, segmentation, or recognition tasks. If it contains actions or trajectories, it may support planning, prediction, or imitation learning.

In [ ]:
def find_label_like_key(example):
    if not isinstance(example, dict):
        return None
    for key in example.keys():
        lower = key.lower()
        if any(token in lower for token in ["label", "class", "target", "action", "goal", "task"]):
            return key
    return None

label_key = find_label_like_key(next(iter(ds)))
print("Suggested label/action key:", label_key)

if label_key is not None:
    counts = []
    for ex in ds.take(2000):
        value = ex[label_key]
        try:
            value = value.numpy()
        except Exception:
            pass
        if isinstance(value, (list, tuple, np.ndarray)):
            counts.append(np.asarray(value).reshape(-1)[0] if np.asarray(value).size > 0 else None)
        elif isinstance(value, (int, float, str, bool)):
            counts.append(value)

    unique_values, counts_values = np.unique(np.asarray([c for c in counts if c is not None], dtype=object), return_counts=True)
    print("Unique label values (sample):")
    for val, cnt in zip(unique_values[:10], counts_values[:10]):
        print(f"  {val}: {cnt}")
else:
    print("No obvious label key detected. The dataset may be more suitable for perception, reinforcement learning, or sequence modeling.")

## Step 5: Estimate the modality

This helps you define a thesis topic more clearly. For example:
- Image/video data -> recognition, tracking, segmentation, scene understanding
- Robot states/actions -> control, planning, imitation learning
- Sensor streams -> time-series forecasting, anomaly detection
- Multimodal data -> perception + language + action reasoning

In [ ]:
def describe_dataset_modality(example):
    if not isinstance(example, dict):
        return "Unknown"
    modalities = []
    for key, value in example.items():
        key_lower = key.lower()
        if "image" in key_lower or "frame" in key_lower or "rgb" in key_lower or "depth" in key_lower:
            modalities.append("visual")
        if "state" in key_lower or "joint" in key_lower or "pose" in key_lower or "vel" in key_lower:
            modalities.append("robot_state")
        if "action" in key_lower or "command" in key_lower or "control" in key_lower:
            modalities.append("action")
        if "audio" in key_lower or "speech" in key_lower:
            modalities.append("audio")
        if "text" in key_lower or "instruction" in key_lower or "prompt" in key_lower:
            modalities.append("text")
    if not modalities:
        return "Could be generic sensor or tabular data"
    return sorted(set(modalities))

print("Detected modalities:", describe_dataset_modality(next(iter(ds))))

## Step 6: How to turn this dataset into a thesis topic

Use the dataset structure to define a problem statement. The best master thesis project usually matches one of these patterns:

### 1. Classification / recognition
- Suitable when the dataset has a label or class per observation
- Example: object recognition, posture recognition, action classification
- Typical title: "Robotic scene understanding using deep learning"

### 2. Detection / segmentation
- Suitable when images or depth maps contain object locations
- Example: object detection, anomaly localization, grasp target detection
- Typical title: "Real-time detection of manipulation targets in robotic perception"

### 3. Sequence prediction / planning
- Suitable when the sample contains time series, states, or action sequences
- Example: next action prediction, task anticipation, trajectory forecasting
- Typical title: "Predictive control and trajectory forecasting for robotic interaction"

### 4. Anomaly / failure detection
- Suitable when normal behavior is dominant and rare events are important
- Example: robotic failure detection, abnormal sensor patterns, unsafe behaviour identification
- Typical title: "Unsupervised anomaly detection for robot operation safety"

### 5. Imitation learning / behavior cloning
- Suitable when observations are paired with expert actions
- Example: learn from demonstrations, map vision to action, predict control commands
- Typical title: "Learning robotic manipulation from demonstrations"

### 6. Multimodal reasoning
- Suitable when the dataset includes language, images, states, and actions together
- Example: instruction-conditioned robot control, semantic command grounding
- Typical title: "Instruction-conditioned robot decision making using multimodal learning"

In [ ]:
def suggest_thesis_direction(example):
    if not isinstance(example, dict):
        return ["General data-driven robotics project"]

    keys = [k.lower() for k in example.keys()]
    suggestions = []

    if any('label' in k or 'class' in k or 'target' in k for k in keys):
        suggestions.append('Classification or recognition project')

    if any('image' in k or 'frame' in k or 'depth' in k or 'mask' in k for k in keys):
        suggestions.append('Perception or detection project')

    if any('action' in k or 'control' in k or 'command' in k or 'trajectory' in k for k in keys):
        suggestions.append('Action prediction or imitation learning project')

    if any('state' in k or 'joint' in k or 'pose' in k or 'sensor' in k for k in keys):
        suggestions.append('Robotics state modeling or control project')

    if any('instruction' in k or 'text' in k or 'prompt' in k for k in keys):
        suggestions.append('Multimodal instruction-following project')

    if not suggestions:
        suggestions.append('Sequential modeling or anomaly detection project')

    return suggestions

print("Suggested thesis directions based on the current sample:")
for item in suggest_thesis_direction(next(iter(ds))):
    print("-", item)

## Step 7: Final decision framework

To choose a strong master thesis topic from this dataset, ask the following questions:

1. Does the dataset contain labels? If yes, classification or recognition is likely.
2. Does it contain images or depth frames? If yes, perception or detection is promising.
3. Does it contain robot actions or trajectories? If yes, predictive control or imitation learning is a good fit.
4. Is the dataset imbalanced or mostly normal cases? If yes, anomaly detection is interesting.
5. Are there multiple modalities together? If yes, multimodal thesis work may be more novel.

A good thesis project is usually not just "use this dataset". It should define a clear research problem, a gap, and a measurable objective.

## Example project titles you can define

Depending on what your dataset contains, you could define title ideas like:

- "Robotic Visual Perception for Object Recognition and Action Understanding"
- "Deep Learning for Task Prediction in Robot Manipulation"
- "Anomaly Detection in Robotic Sensor Streams for Safety Monitoring"
- "Imitation Learning for Robot Control from Demonstration Data"
- "Multimodal Learning for Instruction-conditioned Robot Decision Making"
- "Trajectory Forecasting for Human-Robot Interaction"

In [ ]:
# Final recommendation template
sample = next(iter(ds))
keys = list(sample.keys()) if isinstance(sample, dict) else []

print("Dataset analysis summary:")
print("- Total train examples:", ds.cardinality().numpy())
print("- Sample keys:", keys)
print("- Thesis directions:", suggest_thesis_direction(sample))

print("\nSuggested workflow for your thesis:")
print("1. Write the problem statement based on the main modality in the dataset.")
print("2. Choose one task: classification, detection, forecasting, anomaly detection, or imitation learning.")
print("3. Build a baseline model and compare it with a stronger variant.")
print("4. Evaluate with clear metrics such as accuracy, F1, mAP, MAE, or anomaly recall.")
print("5. Explain why the chosen task matters for robotics or AI.")